<a href="https://colab.research.google.com/github/MoharanaSudhanshu/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MoharanaSudhanshu/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline Rule

The baseline assigns a refresh priority score to each webpage using historical search performance metrics.

Scoring Rule:

• Low CTR (< 2%) → +40 points
• Average Position > 20 → +30 points
• Low Clicks (< 100) → +15 points
• Low Impressions (< 1000) → +10 points
• Content Age > 365 days → +20 points

The final score is the sum of these conditions.

Pages with higher scores receive higher refresh priority.

Reason Codes:

LOW_CTR – Click-through rate is very low.
POOR_POSITION – Average search position is poor.
LOW_CLICKS – Page receives few clicks.
LOW_IMPRESSIONS – Page has low visibility.
OLD_CONTENT – Content is older than one year.

In [2]:
!git clone https://github.com/MoharanaSudhanshu/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 138 (delta 56), reused 91 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.86 MiB | 23.25 MiB/s, done.
Resolving deltas: 100% (56/56), done.
/content/flyrank-ml-internship


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.columns.tolist())
print(df.head())
df["score"] = 0

df.loc[df["ctr"] < 2, "score"] += 40
df["baseline_score"] = (
    (df["trend_pct"] < -20).astype(int) * 30 +
    (df["avg_position"] > 20).astype(int) * 20 +
    (df["ctr"] < 0.03).astype(int) * 15 +
    (df["days_since_last_update"] > 180).astype(int) * 20 +
    (df["clicks_last_30d"] < df["clicks_prev_30d"]).astype(int) * 15
)

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  cont

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

df["baseline_score"] = (
    (df["trend_pct"] < -20).astype(int) * 30 +
    (df["avg_position"] > 20).astype(int) * 20 +
    (df["ctr"] < 0.03).astype(int) * 15 +
    (df["days_since_last_update"] > 180).astype(int) * 20 +
    (df["clicks_last_30d"] < df["clicks_prev_30d"]).astype(int) * 15
)

df = df.sort_values(
    by="baseline_score",
    ascending=False
)

os.makedirs("work/outputs", exist_ok=True)

df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(df[
    [
        "content_id",
        "baseline_score",
        "trend_pct",
        "avg_position",
        "ctr"
    ]
].head(20))

                 content_id  baseline_score  trend_pct  avg_position    ctr
8860   content_adfc46f3f033              85      -60.0          23.7   0.00
505    content_bfa3d6688324              85      -50.0          21.1   0.00
18652  content_0173fb0dc986              85      -92.1          23.0   0.00
7509   content_7a888d3d99c8              85     -100.0          67.6   0.00
15882  content_129753e3095f              85      -50.0          24.0   0.00
29384  content_f6fdf87348f6              85     -100.0          32.5   0.00
698    content_b16bd7307b39              85      -69.7          31.0   0.00
8458   content_b2b2d6b54dda              85      -50.0          21.0  25.00
3507   content_074ba6ead17b              85      -36.5          48.0   0.00
7021   content_1bfaa38ff26c              85      -74.7          22.2   0.23
16514  content_7368877ea310              85      -81.5          24.8   0.13
11489  content_5feee3994adb              85      -89.1          39.0   0.01
26810  conte

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

The highest-ranked pages consistently display multiple negative search performance indicators.

Most pages show:

- Declining traffic trends
- Low CTR
- Poor average search position
- Older content
- Declining clicks compared to the previous month

These pages should be reviewed first because several independent signals suggest they are losing visibility.

The baseline model is transparent and every recommendation can be explained using measurable search metrics.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = df.head(20)

top20[
    [
        "content_id",
        "baseline_score",
        "trend_pct",
        "avg_position",
        "ctr"
    ]
]

,content_id,baseline_score,trend_pct,avg_position,ctr
8860,content_adfc46f3f033,85,-60.0,23.7,0.00
505,content_bfa3d6688324,85,-50.0,21.1,0.00
18652,content_0173fb0dc986,85,-92.1,23.0,0.00
7509,content_7a888d3d99c8,85,-100.0,67.6,0.00
15882,content_129753e3095f,85,-50.0,24.0,0.00
29384,content_f6fdf87348f6,85,-100.0,32.5,0.00
698,content_b16bd7307b39,85,-69.7,31.0,0.00
8458,content_b2b2d6b54dda,85,-50.0,21.0,25.00
3507,content_074ba6ead17b,85,-36.5,48.0,0.00
7021,content_1bfaa38ff26c,85,-74.7,22.2,0.23


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some pages may receive a high score because of temporary traffic fluctuations, seasonal changes, or external search trends.

These pages should always be manually reviewed before making content changes.

## Leakage Check

The baseline rule only uses historical search performance data.

It does not use any future information, prediction labels, or outcome variables.

Therefore, the ranking is free from target leakage and can safely be used as a transparent baseline before training machine learning models.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Leakage Check")

used_columns = [
    "trend_pct",
    "avg_position",
    "ctr",
    "days_since_last_update",
    "clicks_last_30d",
    "clicks_prev_30d"
]

print("Features used:")
for c in used_columns:
    print("-", c)

print("\nNo future information or labels were used.")

Leakage Check
Features used:
- trend_pct
- avg_position
- ctr
- days_since_last_update
- clicks_last_30d
- clicks_prev_30d

No future information or labels were used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.